# Apoptosis Quantification Analysis

This notebook performs automated quantification of apoptosis in 3D acini using cleaved caspase-3 (C3) staining. It segments acini, identifies C3-positive cells, counts total nuclei, and analyzes spatial distribution of apoptotic cells within the acinar structure.

## Analysis Overview
1. Load and rescale 3D fluorescent images
2. Segment acini boundaries
3. Identify C3-positive (apoptotic) cells using watershed segmentation
4. Count total nuclei (DAPI-positive)
5. Calculate spatial metrics and quantify apoptosis per acinus

In [1]:
import numpy as np
import pandas as pd
import os
import pathlib
from joblib import Parallel, delayed
import contextlib
import joblib
from tqdm import tqdm
import warnings

from skimage import util
from skimage.filters import threshold_otsu, gaussian
from skimage.segmentation import clear_border, watershed
from skimage.measure import label, regionprops, regionprops_table
from skimage.morphology import remove_small_holes, remove_small_objects, opening, ball, dilation, erosion
from skimage.transform import rescale
from skimage.feature import peak_local_max

from scipy.ndimage import distance_transform_edt
from scipy import ndimage as ndi

from tifffile import imread
import tifffile

import napari

warnings.filterwarnings("ignore")

c:\Users\isobe\anaconda3\envs\imaging_record\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """
    Enables parallel jobs to run and display of a tqdm progress bar.

    Parameters:
    -----------
    tqdm_object : tqdm
        The tqdm progress bar instance to be updated.
    """
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [3]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Convert an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [4]:
def pixel_size(tif_path):
    """
    Extract pixel size information from TIFF metadata.

    Parameters
    ----------
    tif_path : str
        Path to the TIFF file.

    Returns
    -------
    list
        List containing [x_pixel_size_um, y_pixel_size_um, z_pixel_size_um].
    """
    with tifffile.TiffFile(tif_path) as tif:
        tif_tags = {}
        for tag in tif.pages[0].tags.values():
            name, value = tag.name, tag.value
            tif_tags[name] = value

        x_pixel_size_um = 1/((tif_tags["XResolution"])[0]/(tif_tags["XResolution"][1]))
        y_pixel_size_um = 1/((tif_tags["YResolution"])[0]/(tif_tags["YResolution"][1]))
        try:
            z_pixel_size_um = float(str(tif_tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
        except:
            z_pixel_size_um = (float(str(tif_tags["ImageDescription"]).split("spacing=")[1].split("loop")[0]))
       
    original_spacing = [x_pixel_size_um, y_pixel_size_um, z_pixel_size_um]
    return original_spacing

In [5]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [6]:
def add_image_details(df, filename, flag):
    """
    Add experimental details extracted from the filename to a dataframe.

    This function parses the filename to infer experimental details such as 
    well number, imaging day, mechanical stiffness condition, and treatment type.
    The extracted details are appended as new columns to the dataframe.

    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to which image metadata will be added.
    filename : str
        The filename of the image, used to extract experimental details.
    flag : str
        A flag indicating any segmentation issues detected during processing.

    Returns:
    --------
    df : pandas.DataFrame
        The updated dataframe with the following added columns:
        - 'filename': The original filename.
        - 'flag': Segmentation flag indicating potential issues.
        - 'well': The well number (1 or 2) inferred from the filename.
        - 'day': The experimental time point (0, 1, 3, or 7 days).
        - 'condition': The mechanical stiffness condition ('soft', 'stiff', or 'blank').
        - 'treatment': The treatment applied ('blebbistatin', 'ROCKi', 'batimastat', 'ABT737', or 'none').
        - 'image_type': A combined descriptor of the condition and day (e.g., 'soft, d3').
    """
    df["filename"] = filename
    df["flag"] = flag
    
    # Well number
    if "well1" in filename:
        df["well"] = 1
    else:
        df["well"] = 2
    
    # Day
    if "d0" in filename:
        df["day"] = 0
    elif "d1" in filename:
        df["day"] = 1
    elif "d3" in filename:
        df["day"] = 3
    else:
        df["day"] = 7
    
    # Stiffness
    if 'soft' in filename:
        df["condition"] = 'soft'
    elif 'stiff' in filename:
        df["condition"] = 'stiff'
    else:
        df["condition"] = 'blank'
    
    # Treatment
    if "bleb" in filename.lower():
        df["treatment"] = "blebbistatin"
    elif "rock" in filename.lower():
        df["treatment"] = "ROCKi"
    elif ("batimastat" in filename.lower()) or "mmpi" in filename.lower():
        df["treatment"] = "batimastat"
    elif "abt" in filename.lower():
        df["treatment"] = "ABT737"
    else:
        df["treatment"] = "none"
    
    df['image_type'] = df["condition"].astype(str) + ", d" + df["day"].astype(str)
    return df

In [7]:
def clean_and_watershed(image, acinus_labelled_image, new_pixel_size, separation_in_um=4, radius_threshold_um=2):
    """
    Clean binary mask and perform watershed segmentation to separate individual nuclei.

    Parameters
    ----------
    image : numpy.ndarray
        Binary or intensity image to segment.
    acinus_labelled_image : numpy.ndarray
        Labeled image defining acinus boundaries.
    new_pixel_size : float
        Pixel size in micrometers after rescaling.
    separation_in_um : float, optional
        Minimum distance in micrometers between detected nuclei centers (default: 4).
    radius_threshold_um : float, optional
        Minimum nuclear radius in micrometers for filtering (default: 2).

    Returns
    -------
    watershed_output : numpy.ndarray
        Labeled image with segmented and filtered nuclei.
    """
    cleaned_mask = image * acinus_labelled_image
    cleaned_mask = gaussian(cleaned_mask, 0.8)
    thresh = threshold_otsu(cleaned_mask)
    cleaned_mask = cleaned_mask > thresh
    cleaned_mask = remove_small_objects(cleaned_mask, min_size=1000)
    cleaned_mask = remove_small_holes(cleaned_mask, area_threshold=500)
    cleaned_mask = opening(cleaned_mask, ball(5))
    
    # Watershed nuclei
    distances = ndi.distance_transform_edt(erosion(cleaned_mask, ball(3)))
    nuclear_size_in_pixels = separation_in_um / new_pixel_size
    coordinates = peak_local_max(distances, min_distance=int(nuclear_size_in_pixels))
    marker_locations = coordinates.data
    markers = np.zeros(cleaned_mask.shape, dtype=np.uint32)
    marker_indices = tuple(np.round(marker_locations).astype(int).T)
    markers[marker_indices] = np.arange(len(marker_locations)) + 1
    markers_big = dilation(markers, ball(2))
    segmented = watershed(-distances, markers_big, mask=cleaned_mask)
    segmented = clear_border(segmented)
    
    # Filter out small nuclei
    table = regionprops_table(segmented, properties=('label', 'area'))
    volume_threshold = (4/3) * np.pi * (radius_threshold_um / new_pixel_size)**3
    condition = (table['area'] >= volume_threshold)
    input_labels = table['label']
    output_labels = input_labels * condition
    watershed_output = util.map_array(segmented, input_labels, output_labels)

    return watershed_output

In [8]:
def segment_and_quantify(i, image_paths, c3_segmentation_paths, dapi_segmentation_paths, DAPI_channel=0, C3_channel=3, to_plot=False): 
    """
    Segment acini and quantify C3-positive apoptotic cells.

    This function processes 3D fluorescent images to segment acini, identify C3-positive 
    (apoptotic) cells, count total nuclei, and calculate spatial metrics.

    Parameters
    ----------
    i : int
        Index of the image to process from the paths lists.
    image_paths : list
        List of paths to the raw TIFF image files.
    c3_segmentation_paths : list
        List of paths to C3 binary segmentation masks.
    dapi_segmentation_paths : list
        List of paths to DAPI binary segmentation masks.
    DAPI_channel : int, optional
        Channel index for DAPI (default: 0).
    C3_channel : int, optional
        Channel index for C3 (default: 3).
    to_plot : bool, optional
        If True, return additional arrays for visualization (default: False).

    Returns
    -------
    C3_props : pandas.DataFrame
        DataFrame containing C3-positive cell properties including:
        - label: Cell identifier
        - centroid coordinates (centroid-0, centroid-1, centroid-2)
        - corresponding_distance_matrix: Normalized distance from acinus center
        - C3_volume_um: Volume of C3-positive region in cubic micrometers
        - acinus_vol_in_um: Total acinus volume
        - acinus_roundness: Acinus shape metric
        - number_of_nuclei: Total DAPI-positive nuclei count
        - Experimental metadata (filename, well, day, condition, treatment, etc.)
    
    If to_plot is True, also returns:
        C3_labels, C3_rescaled, distance_scaled, dapi_labels, filtered_labelled_image, 
        acinus_rescaled, rgb_image
    """
    image = convert(imread(image_paths[i]), 0, 255, np.uint8)
    flag = "None"
    filename = str((os.path.basename(image_paths[i]))).lower()
    print(filename)
    
    try:
        # Scale the images
        acinus_image = (image[:, DAPI_channel, :, :] + image[:, C3_channel, :, :])
        
        c3_mask = imread(c3_segmentation_paths[i])
        dapi_mask = imread(dapi_segmentation_paths[i])

        original_spacing = pixel_size(image_paths[i])
        scale_change = original_spacing[2] / original_spacing[0]
        new_pixel_size = 4 * original_spacing[0]
        acinus_rescaled = rescale(scale=(0.25*scale_change, 0.25, 0.25), image=acinus_image, anti_aliasing=False)
        C3_rescaled = rescale(scale=(0.25*scale_change, 0.25, 0.25), image=image[:, C3_channel, :, :], anti_aliasing=False)
        rescaled_c3_mask = rescale(scale=(0.25*scale_change, 0.25, 0.25), image=c3_mask, anti_aliasing=False)
        rescaled_dapi_mask = rescale(scale=(0.25*scale_change, 0.25, 0.25), image=dapi_mask, anti_aliasing=False)

        rescaled_c3_image = rescale(scale=(0.25*scale_change, 0.25, 0.25), image=image[:, C3_channel, :, :], anti_aliasing=False)
        rescaled_dapi_image = rescale(scale=(0.25*scale_change, 0.25, 0.25), image=image[:, DAPI_channel, :, :], anti_aliasing=False)
        rgb_image = np.stack([rescaled_c3_image, np.zeros_like(rescaled_c3_image), rescaled_dapi_image], axis=-1)

        # Clip and pre-process the acinus image to identify boundaries
        clipped = (acinus_rescaled).clip(min=np.quantile(acinus_rescaled, 0.05), max=np.quantile(acinus_rescaled, 0.85))
        acinus_smoothed = gaussian(clipped, sigma=12)
        thresh = threshold_otsu(acinus_smoothed)
        binary = acinus_smoothed > thresh
        binary = remove_small_holes(binary, area_threshold=100000)
        binary = remove_small_objects(binary, min_size=10000)
        binary = erosion(binary, ball(5))
        labelled_image = label(binary)
        table = regionprops_table(labelled_image, properties=('label', 'area'))
        condition = (table['area'] >= table['area'].max())
        # Filter the labels so we only keep the largest one
        input_labels = table['label']
        output_labels = input_labels * condition
        filtered_labelled_image = util.map_array(labelled_image, input_labels, output_labels)
        acinus_vol_in_um = [(region.area) * (new_pixel_size**3) for region in regionprops(filtered_labelled_image)][0]
        acinus_roundness = [(region.inertia_tensor_eigvals[2] / region.inertia_tensor_eigvals[0]) for region in regionprops(filtered_labelled_image)][0]

        # Identify C3* +ve objects and all nuclei
        C3_labels = clean_and_watershed(rescaled_c3_mask, filtered_labelled_image, new_pixel_size, separation_in_um=7, radius_threshold_um=1.3)
        dapi_labels = clean_and_watershed(rescaled_dapi_mask, filtered_labelled_image, new_pixel_size, separation_in_um=6, radius_threshold_um=2)

        distance = (distance_transform_edt(binary)) * (new_pixel_size * new_pixel_size)
        distance_scaled = np.interp(distance, (distance.min(), distance.max()), (0, +1))

        if pd.DataFrame(regionprops_table(C3_labels, properties=("label", "area", "centroid"))).shape[0] == 0:
            C3_props = pd.DataFrame(columns=["label", "centroid-0", "centroid-1", "centroid-2", "corresponding_distance_matrix", "C3_volume_um"], 
                                   data=[["no_c3", np.NaN, np.NaN, np.NaN, np.NaN, np.NaN]])
            C3_props["filename"] = filename
        else:
            C3_props = pd.DataFrame(regionprops_table(C3_labels, properties=("label", "area", "centroid")))
            C3_props["corresponding_distance_matrix"] = C3_props.apply(lambda x: distance_scaled[round(x['centroid-0']), round(x['centroid-1']), round(x['centroid-2'])], axis=1)
            C3_props["C3_volume_um"] = C3_props["area"] * new_pixel_size**3
            C3_props.drop('area', axis=1, inplace=True)
            C3_props["filename"] = filename
        
        C3_props["acinus_vol_in_um"] = acinus_vol_in_um
        C3_props["acinus_roundness"] = acinus_roundness
        C3_props = add_image_details(C3_props, filename, flag)

        dapi_info = pd.DataFrame(regionprops_table(dapi_labels, properties=("label", "area")))
        C3_props["number_of_nuclei"] = dapi_info.shape[0]

        if to_plot:
            return C3_props, C3_labels, C3_rescaled, distance_scaled, dapi_labels, filtered_labelled_image, acinus_rescaled, rgb_image
    
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        C3_props = pd.DataFrame(columns=["label", "centroid-0", "centroid-1", "centroid-2", "corresponding_distance_matrix", "C3_volume_um"], 
                               data=[["Fail", "Fail", "Fail", "Fail", "Fail", "Fail"]])
        C3_props["filename"] = filename
        C3_props["flag"] = "FAILED"
    
    return C3_props

## Single Image Analysis

Test the analysis pipeline on a single image to verify segmentation quality and parameters.

In [9]:
# Define paths to image directories
root = r"E:\Fabianna_N2_Images\integrin_analysis"
image_root = root + "\\tif"
c3_root = root + "\\combined_binary_c3"
dapi_root = root + "\\combined_binary_dapi"

# Get all image paths
image_paths = list(pathlib.Path(image_root).glob("**/*.tif"))
c3_paths = list(pathlib.Path(c3_root).glob("**/*.tif"))
dapi_paths = list(pathlib.Path(dapi_root).glob("**/*.tif"))

print(f"Found {len(image_paths)} images, {len(c3_paths)} C3 masks, {len(dapi_paths)} DAPI masks")

Found 20 images, 20 C3 masks, 20 DAPI masks


In [10]:
# Process a single image for testing
i = 7
C3_props, C3_labels, C3_rescaled, distance_scaled, dapi_labels, filtered_labelled_image, acinus_image, rgb_image = segment_and_quantify(
    i, image_paths, c3_paths, dapi_paths, C3_channel=2, to_plot=True
)

d7_soft_12g10_c3008_cmle.tif


In [11]:
C3_props

,label,centroid-0,centroid-1,centroid-2,corresponding_distance_matrix,C3_volume_um,filename,acinus_vol_in_um,acinus_roundness,flag,well,day,condition,treatment,image_type,number_of_nuclei
0,1,102.210627,137.112073,139.050105,0.623916,2845.72217,d7_soft_12g10_c3008_cmle.tif,69486.912595,0.893793,None,2,7,soft,none,"soft, d7",30


In [26]:
# Visualize segmentation results in Napari
viewer = napari.Viewer()
viewer.add_image(C3_rescaled, name='C3 Channel')
viewer.add_image(acinus_image, name='Acinus Image')
viewer.add_labels(C3_labels, name='C3 Labels')
viewer.add_labels(dapi_labels, name='DAPI Labels')
viewer.add_labels(filtered_labelled_image, name='Acinus Mask')
viewer.add_image(rgb_image, name='RGB Composite')

<Image layer 'RGB Composite' at 0x1939c1e2530>

In [105]:
C3_props

,label,centroid-0,centroid-1,centroid-2,corresponding_distance_matrix,C3_volume_um,filename,acinus_vol_in_um,flag,well,day,condition,image_type,number_of_nuclei
0,1,124.581940,138.009197,69.312034,0.652011,1506.038788,ecadc3d1softwell11p8zoom002interestingecadnext...,162844.877133,None,1,1,soft,"soft, d1",42
1,2,87.888060,143.875081,107.936080,0.585990,344.269352,ecadc3d1softwell11p8zoom002interestingecadnext...,162844.877133,None,1,1,soft,"soft, d1",42
2,3,124.682227,107.838972,102.240685,0.511517,130.413520,ecadc3d1softwell11p8zoom002interestingecadnext...,162844.877133,None,1,1,soft,"soft, d1",42
3,4,142.388263,214.484884,134.603438,0.190086,94.221674,ecadc3d1softwell11p8zoom002interestingecadnext...,162844.877133,None,1,1,soft,"soft, d1",42
4,5,122.390390,210.251251,133.251251,0.219870,55.795763,ecadc3d1softwell11p8zoom002interestingecadnext...,162844.877133,None,1,1,soft,"soft, d1",42


## Batch Processing

Process all images in parallel to generate the complete dataset.

In [ ]:
# Process all images in parallel
with tqdm_joblib(tqdm(desc="Image Analysis", total=len(image_paths))) as progress_bar:
    c3_props = Parallel(n_jobs=4)(
        delayed(segment_and_quantify)(i, image_paths, c3_paths, dapi_paths, to_plot=False) 
        for i in range(len(image_paths))
    )

# Combine results into a single DataFrame
results_df = pd.concat(c3_props, ignore_index=True)
print(f"\nAnalysis complete! Processed {len(image_paths)} images.")
print(f"Total C3-positive cells detected: {results_df[results_df['label'] != 'no_c3'].shape[0]}")

Image Analysis: 100%|██████████| 157/157 [48:22<00:00, 18.49s/it]
